# AWS SageMaker - Triton Inference Server via Multi-Model Endpoint (MME)

This notebook deploys ONNX models to **NVIDIA Triton Inference Server** running on an AWS SageMaker Multi-Model Endpoint.

## Architecture

SageMaker MME works **fundamentally differently** from Azure and GCP:

| Resource | Purpose |
|---|---|
| **Model** | Points to an S3 *prefix* and the Triton Docker image. `Mode: MultiModel` tells SageMaker to treat the prefix as a collection of independently loadable models. |
| **EndpointConfig** | Machine specification (`ml.g4dn.xlarge` with T4 GPU) - just configuration, no compute yet. |
| **Endpoint** | The resource that actually provisions the GPU instance(s) - the main billable component. |

### Key difference from Azure / GCP

- Models do **not** need to be present at startup. SageMaker MME dynamically discovers new `.tar.gz` archives in the S3 prefix and loads them on demand via `TargetModel`.
- Each model is packaged as a separate tarball.
- Adding a new model = uploading a new tarball to S3 - **no redeployment**.

## Prerequisites

- AWS account with SageMaker, S3, IAM, and CloudWatch permissions.
- A SageMaker execution role with S3 access.
- Local model directories under `./models3/`.
- Python packages: `boto3`, `sagemaker`, `numpy`, `pillow`, `transformers`.

## 1. Imports

In [ ]:
import json
import random
import string
import subprocess
import time
from pathlib import Path

import boto3
import numpy as np
from PIL import Image
from sagemaker.core.helper.session_helper import Session, get_execution_role
from transformers import AutoConfig, AutoImageProcessor

## 2. Configuration

In [ ]:
SAGEMAKER_TRITON_VERSION = "24.09"
# https://aws.github.io/deep-learning-containers/reference/region_availability/
TRITON_ACCOUNT_ID = "763104351884"

INSTANCE_TYPE = "ml.g4dn.xlarge"
LOCAL_MODELS_DIR = "./models3"
ARTIFACTS_DIR = "./artifacts"
TEST_IMAGE_PATH = "./test_image.jpg"

# Each entry maps (triton model dir name) -> (tarball filename)
MODEL_TARBALLS = {
    "resnet50_onnx": "resnet50.tar.gz",
    "convnext_tiny_onnx": "convnext_tiny.tar.gz",
    "mobilenet_v2_onnx": "mobilenet_v2.tar.gz",
}

MODELS = {
    "resnet50_onnx": {
        "hf_model": "microsoft/resnet-50",
        "input_name": "pixel_values",
        "output_name": "logits",
    },
    "convnext_tiny_onnx": {
        "hf_model": "facebook/convnext-tiny-224",
        "input_name": "pixel_values",
        "output_name": "logits",
    },
    "mobilenet_v2_onnx": {
        "hf_model": "google/mobilenet_v2_1.0_224",
        "input_name": "pixel_values",
        "output_name": "logits",
        "processor_kwargs": {"crop_size": {"height": 256, "width": 256}},
    },
}

## 3. Helpers

In [ ]:
def softmax(logits: np.ndarray) -> np.ndarray:
    exp = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
    return exp / np.sum(exp, axis=-1, keepdims=True)


def decode_top1(logits: np.ndarray, hf_model: str) -> tuple[str, float]:
    config = AutoConfig.from_pretrained(hf_model)
    probs = softmax(logits)
    pred_idx = int(np.argmax(probs, axis=-1)[0])
    score = float(probs[0, pred_idx])
    label = config.id2label.get(pred_idx, f"class_{pred_idx}")
    return label, score


def triton_image_uri(region: str) -> str:
    return (
        f"{TRITON_ACCOUNT_ID}.dkr.ecr.{region}.amazonaws.com"
        f"/sagemaker-tritonserver:{SAGEMAKER_TRITON_VERSION}-py3"
    )

## 4. Initialize AWS Session

In [ ]:
session = boto3.Session()
region = session.region_name or "eu-west-1"
sm_client = session.client("sagemaker")
sagemaker_session = Session(boto_session=session)

role_arn = get_execution_role(sagemaker_session)
default_bucket = sagemaker_session.default_bucket()

suffix = "".join(random.choices(string.ascii_lowercase + string.digits, k=8))
model_name = f"triton-densenet-{suffix}"
endpoint_config_name = f"triton-densenet-cfg-{suffix}"
endpoint_name = f"triton-densenet-endpoint-{suffix}"
s3_key_prefix = f"triton/single-model/{suffix}"

print(f"Region:          {region}")
print(f"Role:            {role_arn}")
print(f"Default bucket:  {default_bucket}")
print(f"Endpoint name:   {endpoint_name}")

## 5. Package & Upload Models to S3

Each model is packaged as a separate `.tar.gz` tarball and uploaded to S3. SageMaker MME will dynamically discover these tarballs.

In [ ]:
# Package each model as a separate tarball
Path(ARTIFACTS_DIR).mkdir(parents=True, exist_ok=True)

for model_dir_name, tar_name in MODEL_TARBALLS.items():
    tar_path = Path(ARTIFACTS_DIR) / tar_name
    subprocess.run(
        ["tar", "-czvf", str(tar_path), "-C", LOCAL_MODELS_DIR, model_dir_name],
        check=True,
    )

In [ ]:
# Upload all tarballs to S3
for tar_file in Path(ARTIFACTS_DIR).glob("*.tar.gz"):
    sagemaker_session.upload_data(
        path=str(tar_file),
        bucket=default_bucket,
        key_prefix=s3_key_prefix,
    )

## 6. Create SageMaker Model

Points to the S3 prefix (with trailing `/`) so MME discovers all tarballs. `Mode: MultiModel` enables dynamic model loading.

In [ ]:
container = {
    "Image": triton_image_uri(region),
    "ModelDataUrl": f"s3://{default_bucket}/{s3_key_prefix}/",
    "Mode": "MultiModel",
}

create_model_response = sm_client.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role_arn,
    PrimaryContainer=container,
)
print(f"ModelArn: {create_model_response['ModelArn']}")

## 7. Create Endpoint Config & Endpoint

- **EndpointConfig** - machine specification only, no compute yet
- **Endpoint** - provisions GPU instance (the main billable resource)

In [ ]:
# Endpoint config (machine specification only - no compute yet)
create_endpoint_config_response = sm_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InitialInstanceCount": 1,
            "InstanceType": INSTANCE_TYPE,
            "InitialVariantWeight": 1.0,
        }
    ],
)
print(f"EndpointConfigArn: {create_endpoint_config_response['EndpointConfigArn']}")

In [ ]:
# Create endpoint (provisions GPU instance - the main billable resource)
create_endpoint_response = sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
)
print(f"EndpointArn: {create_endpoint_response['EndpointArn']}")

In [ ]:
# Wait for the endpoint to become InService
status = "Creating"
while status == "Creating":
    response = sm_client.describe_endpoint(EndpointName=endpoint_name)
    status = response["EndpointStatus"]
    print(f"Endpoint status: {status}")
    if status == "Creating":
        time.sleep(60)

if status != "InService":
    raise RuntimeError(f"Endpoint deployment failed with status: {status}")

## 8. Inference

Unlike Azure and GCP (which address models via Triton's model name), SageMaker MME uses `TargetModel` - the tarball filename in S3. SageMaker dynamically loads the model on first request.

In [ ]:
runtime_sm_client = session.client("sagemaker-runtime")
img = Image.open(TEST_IMAGE_PATH).convert("RGB")

for model_name_triton, tar_name in MODEL_TARBALLS.items():
    cfg = MODELS[model_name_triton]
    processor_kwargs = cfg.get("processor_kwargs", {})
    processor = AutoImageProcessor.from_pretrained(cfg["hf_model"], **processor_kwargs)
    inputs = processor(images=img, return_tensors="np")
    pixel_values = inputs["pixel_values"].astype(np.float32)

    # Build Triton V2 JSON payload (same format as GCP)
    payload = {
        "inputs": [
            {
                "name": cfg["input_name"],
                "shape": list(pixel_values.shape),
                "datatype": "FP32",
                "data": pixel_values.flatten().tolist(),
            }
        ],
        "outputs": [{"name": cfg["output_name"]}],
    }

    # TargetModel selects which tarball to load from the S3 prefix
    response = runtime_sm_client.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Body=json.dumps(payload),
        TargetModel=tar_name,
    )

    result = json.loads(response["Body"].read().decode("utf-8"))
    logits = np.array(result["outputs"][0]["data"]).reshape(1, -1)

    label, score = decode_top1(logits, cfg["hf_model"])
    print(f"{model_name_triton:<30s}  label={label:<40s}  score={score:.4f}")

## 9. Clean up resources

Run this section when you no longer need the endpoint.

In [ ]:
delete = False

In [ ]:
if delete:
    sm_client.delete_endpoint(EndpointName=endpoint_name)

    while True:
        try:
            response = sm_client.describe_endpoint(EndpointName=endpoint_name)
            print(f"Endpoint delete status: {response['EndpointStatus']}")
            time.sleep(30)
        except sm_client.exceptions.ClientError as error:
            if "Could not find endpoint" in str(error):
                print("Endpoint deleted")
                break
            raise

    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    sm_client.delete_model(ModelName=model_name)
    print("Cleanup complete")